# Diagnóstico de Captura de Áudio no Windows
## Solução para erro: "Verifique nas configurações de som do Windows se o microfone correto está selecionado"

Este notebook ajuda a identificar e resolver problemas de captura de áudio no ClinicaFlow, especialmente com fones Bluetooth no Windows.

## Seção 1: Preparar Ambiente e Dependências

Primeiro, instalamos as bibliotecas necessárias para diagnosticar dispositivos de áudio.

In [ ]:
import subprocess
import sys

# Instalar sounddevice (gerenciador de dispositivos de áudio multiplataforma)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sounddevice", "numpy"])

print("✅ Dependências instaladas com sucesso!")

In [ ]:
import sounddevice as sd
import numpy as np
import platform
import subprocess as sp

print(f"Sistema Operacional: {platform.system()} {platform.version()}")
print(f"Python: {platform.python_version()}")
print(f"sounddevice: {sd.__version__}")

## Seção 2: Listar Dispositivos de Entrada de Áudio

Localiza todos os microfones disponíveis no computador.

In [ ]:
# Listar todos os dispositivos de áudio
devices = sd.query_devices()
print(f"Total de dispositivos de áudio: {len(devices)}\n")
print("📱 DISPOSITIVOS DE ENTRADA (Microfones):")
print("-" * 80)

input_devices = []
for idx, device in enumerate(devices):
    # Filtrar apenas entradas (input)
    if device['max_input_channels'] > 0:
        input_devices.append((idx, device))
        print(f"[{idx}] {device['name']}")
        print(f"    Canais: {device['max_input_channels']}")
        print(f"    Sample Rate: {device['default_samplerate']} Hz")
        print(f"    Latência de entrada: {device['default_low_input_latency']*1000:.1f}ms")
        print()

if not input_devices:
    print("❌ Nenhum dispositivo de entrada encontrado!")
else:
    print(f"✅ {len(input_devices)} dispositivo(s) de entrada disponível(is)")

## Seção 3: Detectar o Microfone Padrão do Sistema

Identifica qual dispositivo está configurado como padrão no Windows.

In [ ]:
# Detectar dispositivo de entrada padrão
default_device_idx = sd.default.device[0]  # [0] = input, [1] = output
default_device = devices[default_device_idx]

print("🎤 DISPOSITIVO DE ENTRADA PADRÃO (ativo agora):")
print("-" * 80)
print(f"Índice: [{default_device_idx}]")
print(f"Nome: {default_device['name']}")
print(f"Canais: {default_device['max_input_channels']}")
print(f"Sample Rate: {default_device['default_samplerate']} Hz")

# Verificar se o padrão é Bluetooth ou fone
is_headset = any(keyword in default_device['name'].lower() for keyword in ['headset', 'headphones', 'bluetooth', 'bt', 'wireless', 'buds'])
is_integrated = 'microphone' in default_device['name'].lower() or 'stereo mix' in default_device['name'].lower()

if is_headset:
    print("✅ Tipo: Fone/Headset (Possivelmente Bluetooth)")
elif is_integrated:
    print("✅ Tipo: Microfone integrado do notebook")
else:
    print("ℹ️ Tipo: Outro dispositivo")

## Seção 4: Teste de Gravação por Dispositivo

Grava 3 segundos de áudio em cada microfone e mede o nível de energia (RMS) para verificar se há sinal válido.

In [ ]:
import time

DURATION = 3  # segundos
test_results = []

print("🎙️ TESTE DE GRAVAÇÃO (3 segundos em cada microfone)")
print("⚠️ Fale alguma coisa durante a gravação!\n")
print("-" * 80)

for idx, device in input_devices:
    device_name = device['name']
    sample_rate = int(device['default_samplerate'])
    
    print(f"▶️ Gravando de [{idx}] {device_name}... ", end="", flush=True)
    
    try:
        # Gravar áudio
        audio_data = sd.rec(int(DURATION * sample_rate), samplerate=sample_rate, channels=1, device=idx)
        sd.wait()
        
        # Calcular energia (RMS)
        rms = np.sqrt(np.mean(audio_data**2))
        peak = np.max(np.abs(audio_data))
        
        # Limiar para detectar áudio significativo
        # avgEnergy < 0.005 no frontend significa muito silêncio
        is_valid = rms > 0.005
        status = "✅ VÁLIDO" if is_valid else "❌ SILÊNCIO"
        
        print(f"\n  RMS: {rms:.6f} | Peak: {peak:.6f} | {status}")
        
        test_results.append({
            'index': idx,
            'name': device_name,
            'sample_rate': sample_rate,
            'rms': rms,
            'peak': peak,
            'is_valid': is_valid
        })
        
    except Exception as e:
        print(f"❌ Erro: {e}")

print("\n" + "=" * 80)

## Seção 5: Comparação Microfone Integrado vs Fone com Fio

Analisa lado a lado qual dispositivo tem melhor captura de áudio.

In [ ]:
# Classificar dispositivos
print("📊 RESUMO DOS RESULTADOS:")
print("-" * 80)

integrated_results = [r for r in test_results if 'microphone' in r['name'].lower()]
headset_results = [r for r in test_results if any(kw in r['name'].lower() for kw in ['headset', 'headphones', 'bluetooth', 'wireless', 'buds'])]

def print_comparison(device_type, results):
    if not results:
        print(f"\n{device_type}: Não encontrado")
        return
    
    print(f"\n{device_type}:")
    for r in results:
        status_emoji = "✅" if r['is_valid'] else "❌"
        print(f"  {status_emoji} [{r['index']}] {r['name']}")
        print(f"      RMS: {r['rms']:.6f} (Limite: 0.005)")
        print(f"      SR: {r['sample_rate']} Hz")

print_comparison("🎙️ Microfone Integrado", integrated_results)
print_comparison("🎧 Fone/Headset", headset_results)

# Recomendação
print("\n" + "=" * 80)
print("💡 RECOMENDAÇÃO:")
valid_devices = [r for r in test_results if r['is_valid']]
if valid_devices:
    best = max(valid_devices, key=lambda x: x['rms'])
    print(f"✅ Use o dispositivo de entrada: [{best['index']}] {best['name']}")
    print(f"   (Energia RMS: {best['rms']:.6f})")
else:
    print("❌ Nenhum dispositivo com captura de áudio válida foi encontrado!")
    print("   Possíveis causas:")
    print("   • Microfone mudo ou desconectado")
    print("   • Fone Bluetooth no perfil errado (A2DP em vez de Hands-Free/Headset)")
    print("   • Permissão de microfone negada para aplicações")

## Seção 6: Ações Recomendadas

Siga os passos abaixo se o teste falhar ou tiver a energia muito baixa.

### 🔧 Se você recebe "Verifique nas configurações de som do Windows..."

Este erro aparece quando a energia do áudio capturado está **muito baixa** (RMS ≤ 0.005).

#### Passo 1: Verificar Dispositivo Padrão
1. Clique com botão direito no ícone de volume (canto inferior direito)
2. Selecione "Configurações de som"
3. Role para baixo até "Dispositivos de entrada avançada"
4. Verifique qual dispositivo está selecionado em "Dispositivo de entrada padrão"

#### Passo 2: Se está usando Fone Bluetooth
❌ **Problema comum**: O Windows deixa o fone em modo de áudio (A2DP = só saída) em vez de voz (HFP = entrada+saída)

✅ **Solução**:
1. Abra "Configurações de som"
2. Role até "Dispositivos avançados"
3. Procure por seu dispositi Bluetooth (ex: "Headphones - Bluetooth Audio")
4. Clique nele e procure por um seletor de "Tipo de conexão" ou "Protocolo"
5. Mude para **"Hands-Free Headset"** ou **"Headset (Voice)"**

Isso ativa o perfil HFP (voz) que permite captura de áudio.

#### Passo 3: Se está usando Fone com Fio
✅ Fone com fio geralmente funciona bem, mas confirme:
1. O fone está conectado à entrada correta (verde ou com ícone de microfone)
2. Não está mudo (verifique o mute/volume do fone)
3. No Windows, selecione este dispositivo como padrão

#### Passo 4: Se está usando Microfone Integrado
✅ Geralmente é mais confiável que Bluetooth:
1. Verifique se não está mudo (procure por botão de mute no notebook)
2. Aumente o ganho: Configurações de Som → Dispositivo de entrada → Volume 100%
3. Desabilite "Cancelamento de eco" se a captura ficar ruim

#### Passo 5: Testar se Resolveu
Rode o teste de gravação desta célula novamente e verifique se a energia (RMS) agora é > 0.005

### 🎯 Resumo e Próximos Passos

1. **Teste novamente após ajustar**: Rode a célula de gravação novamente para confirmar que a energia agora está acima de 0.005

2. **Se ainda não funciona com Bluetooth**:
   - Desconecte e reconecte o dispositivo Bluetooth
   - Abra "Gerenciador de Dispositivos" (Win+X → Gerenciador de Dispositivos)
   - Procure por "Dispositivos de Áudio" → Clique direito no Bluetooth → Atualizar driver

3. **Mudou de dispositivo?** Teste direto no ClinicaFlow:
   - Vá para o prontuário e clique em "Modo Voz"
   - Fale uma frase clara durante a gravação
   - Se funcionar, deve gerar transcrição e SOAP automaticamente

4. **Ainda dá erro no ClinicaFlow?** Contribua com logs coletando:
   - Abra DevTools (F12) → Console
   - Na próxima tentativa de gravação, procure por linhas com `[AudioRecorder]` ou `[VoiceSOAP]`
   - Copie e compartilhe os logs para diagnóstico aprofundado